# 01 — Nettoyage des données

On part des fichiers bruts Binance (`data/raw/BTCUSDT-1h-*.zip`) et on produit un dataset propre (`data/processed/btc_clean.csv`) prêt pour les features.

Objectif : bons types, colonnes utiles, **trié par date croissante**, sans doublons ni trous horaires.

## Partie 1 — Chargement des fichiers bruts

Les données Binance arrivent en 6 fichiers `.zip` mensuels, **sans en-tête**. On les lit un par un, on leur donne les bons noms de colonnes, puis on les empile en un seul dataset.

In [1]:
import pandas as pd


In [2]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

files = sorted(RAW_DIR.glob("BTCUSDT-1h-*.zip"))

print(f"Nombre de fichiers trouvés : {len(files)}")

for f in files:
    print(f.name)

Nombre de fichiers trouvés : 6
BTCUSDT-1h-2025-12.zip
BTCUSDT-1h-2026-01.zip
BTCUSDT-1h-2026-02.zip
BTCUSDT-1h-2026-03.zip
BTCUSDT-1h-2026-04.zip
BTCUSDT-1h-2026-05.zip


In [3]:
columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

In [5]:
dfs = []

In [6]:
for file in files:
    temp_df = pd.read_csv(file, header=None)
    temp_df.columns = columns
    dfs.append(temp_df)
df = pd.concat(dfs, ignore_index=True)
print("Shape du dataset :", df.shape)
df.head()

Shape du dataset : (4368, 12)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1764547200000000,90360.01,90417.00,86959.99,87000.00,4607.45526,1764550799999999,4.075802e+08,742069,1881.95608,1.663911e+08,0
1,1764550800000000,87000.00,87749.99,86941.40,87168.91,1588.04096,1764554399999999,1.387834e+08,368794,865.41630,7.563708e+07,0
2,1764554400000000,87168.90,87500.00,86474.34,86722.29,2052.19788,1764557999999999,1.784774e+08,306696,840.04871,7.306030e+07,0
3,1764558000000000,86722.30,86800.00,86161.61,86346.13,2001.96556,1764561599999999,1.730082e+08,305414,603.75053,5.218520e+07,0
4,1764561600000000,86346.13,86350.01,85695.44,85801.03,1863.01351,1764565199999999,1.602050e+08,290041,731.14896,6.286958e+07,0


## Partie 2 — Conversion des types

Les colonnes de temps sont en microsecondes (entiers) → on les passe en `datetime`. On force aussi les prix/volumes en `float` et `number_of_trades` en `int`.

In [8]:
df["open_time"] = pd.to_datetime(df["open_time"], unit="us")
df["close_time"] = pd.to_datetime(df["close_time"], unit="us")

df[["open_time", "close_time"]].head()

,open_time,close_time
0,2025-12-01 00:00:00,2025-12-01 00:59:59.999999
1,2025-12-01 01:00:00,2025-12-01 01:59:59.999999
2,2025-12-01 02:00:00,2025-12-01 02:59:59.999999
3,2025-12-01 03:00:00,2025-12-01 03:59:59.999999
4,2025-12-01 04:00:00,2025-12-01 04:59:59.999999


In [9]:
df.dtypes

open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
ignore                             int64
dtype: object

In [10]:
float_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "taker_buy_base_volume",
    "taker_buy_quote_volume"
]

df[float_cols] = df[float_cols].astype(float)

In [11]:
df["number_of_trades"] = df["number_of_trades"].astype(int)

## Partie 3 — Contrôles & nettoyage

On vérifie que les heures se suivent (écart régulier de 1h), on supprime la colonne `ignore` (inutile), on **trie par date croissante** (critique) et on retire les doublons d'horodatage.

In [13]:
time_diff = df["open_time"].diff()

time_diff.value_counts()

open_time
0 days 01:00:00    4367
Name: count, dtype: int64

In [14]:
df = df.drop(columns=["ignore"])

In [15]:
df.columns

Index(['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
       'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume',
       'taker_buy_quote_volume'],
      dtype='str')

In [16]:
df = df.sort_values("open_time").reset_index(drop=True)

In [17]:
df = df.drop_duplicates(subset=["open_time"]).reset_index(drop=True)

## Partie 4 — Vérifications finales

Dernier contrôle avant sauvegarde : aucun `NaN`, dates bien triées, bons types, et aucun trou horaire.

In [18]:
df.isna().sum()

open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
dtype: int64

In [19]:
df["open_time"].is_monotonic_increasing

True

In [20]:
df.dtypes

open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

In [21]:
missing_hours = df[time_diff > pd.Timedelta(hours=1)]

missing_hours[["open_time"]]

,open_time


## Partie 5 — Sauvegarde

Le dataset propre est enregistré dans `data/processed/btc_clean.csv`. C'est le point de départ des features (Partie 2 du projet, faite par Lina).

In [23]:
df.to_csv("../data/processed/btc_clean.csv", index=False)

print("Dataset sauvegardé")

Dataset sauvegardé


In [24]:
print("Shape:", df.shape)
print("\nColonnes:")
print(df.columns.tolist())

print("\nTypes:")
print(df.dtypes)

print("\nValeurs manquantes:")
print(df.isna().sum())

print("\nDoublons open_time:", df.duplicated(subset=["open_time"]).sum())

print("\nDates triées:", df["open_time"].is_monotonic_increasing)

print("\nPremières lignes:")
display(df.head())

print("\nDernières lignes:")
display(df.tail())

Shape: (4368, 11)

Colonnes:
['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume']

Types:
open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

Valeurs manquantes:
open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
dtype:

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2025-12-01 00:00:00,90360.01,90417.00,86959.99,87000.00,4607.45526,2025-12-01 00:59:59.999999,4.075802e+08,742069,1881.95608,1.663911e+08
1,2025-12-01 01:00:00,87000.00,87749.99,86941.40,87168.91,1588.04096,2025-12-01 01:59:59.999999,1.387834e+08,368794,865.41630,7.563708e+07
2,2025-12-01 02:00:00,87168.90,87500.00,86474.34,86722.29,2052.19788,2025-12-01 02:59:59.999999,1.784774e+08,306696,840.04871,7.306030e+07
3,2025-12-01 03:00:00,86722.30,86800.00,86161.61,86346.13,2001.96556,2025-12-01 03:59:59.999999,1.730082e+08,305414,603.75053,5.218520e+07
4,2025-12-01 04:00:00,86346.13,86350.01,85695.44,85801.03,1863.01351,2025-12-01 04:59:59.999999,1.602050e+08,290041,731.14896,6.286958e+07



Dernières lignes:


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
4363,2026-05-31 19:00:00,73603.68,73649.93,73456.48,73563.42,255.80848,2026-05-31 19:59:59.999999,1.881420e+07,44699,152.82907,1.124137e+07
4364,2026-05-31 20:00:00,73563.41,73760.00,73563.41,73739.66,381.56410,2026-05-31 20:59:59.999999,2.810602e+07,46998,247.93090,1.826117e+07
4365,2026-05-31 21:00:00,73739.67,73924.00,73626.67,73892.52,391.01047,2026-05-31 21:59:59.999999,2.883900e+07,82920,232.88775,1.717476e+07
4366,2026-05-31 22:00:00,73892.52,73997.56,73686.00,73944.62,284.29172,2026-05-31 22:59:59.999999,2.101066e+07,100294,164.47721,1.215668e+07
4367,2026-05-31 23:00:00,73944.62,74198.00,73500.00,73674.39,375.30928,2026-05-31 23:59:59.999999,2.771456e+07,110857,179.10216,1.323043e+07


In [25]:
Path("../data/processed/btc_clean.csv").exists()

True